In [1]:
import matplotlib.pyplot as plt
import numpy as np
import os
import sys
import pickle

path = os.getcwd().split(os.sep + 'GUI')[0]
if path not in sys.path:
    print("not here")
    sys.path.append(path)

from neurolib.models.aln import ALNModel
from neurolib.utils import plotFunctions as plotFunc
from neurolib.utils import costFunctions as cost
import neurolib.dashboard.functions as functions
import neurolib.dashboard.data as data
    
# This will reload all imports as soon as the code changes
%load_ext autoreload
%autoreload 2 

#path = os.path.join(os.getcwd(), "plots")

not here


In [2]:
# read case
print(os.getcwd())
case = os.getcwd().split(os.sep)[-1]
print(case[0])

/mnt/antares_raid/home/salfenmoser/neurolib/GUI/current/gui/data/00000
0


### Bistability

In [3]:
aln = ALNModel()
N = aln.params.N

data.set_parameters(aln)

state_vars = aln.state_vars
init_vars = aln.init_vars

##############################################################
def setinit(init_vars_, model):
    state_vars = model.state_vars
    init_vars = model.init_vars
    for iv in range(len(init_vars)):
        for sv in range(len(state_vars)):
            if state_vars[sv] in init_vars[iv]:
                #print("set init vars ", )
                if model.params[init_vars[iv]].ndim == 2:
                    model.params[init_vars[iv]][0,:] = init_vars_[sv]
                else:
                    model.params[init_vars[iv]][0] = init_vars_[sv]
                    
##############################################################               
def setmaxmincontrol(max_c_c, min_c_c, max_c_r, min_c_r):
    import numpy as np
    
    max_cntrl = np.zeros(( 6 ))
    min_cntrl = np.zeros(( 6 ))
    
    max_cntrl[0] = max_c_c
    min_cntrl[0] = min_c_c
    max_cntrl[1] = max_c_c
    min_cntrl[1] = min_c_c
    max_cntrl[2] = max_c_r
    min_cntrl[2] = min_c_r
    max_cntrl[3] = max_c_r
    min_cntrl[3] = min_c_r
    max_cntrl[4] = max_c_r
    min_cntrl[4] = min_c_r
    max_cntrl[5] = max_c_r
    min_cntrl[5] = min_c_r
            
    return max_cntrl, min_cntrl

#####################################################
def getclosest(k_, found_solution, exc, inh, already_tried_):
    import numpy as np
    if len(found_solution) == 0:
        print("no solutions found")
        return -1
    
    start_ind = -1
    for j_ in found_solution:
        if j_ not in already_tried_ and j_ != k_:
            start_ind = j_
            break
            
    if start_ind == -1:
        return -1
        
    min_dist = np.sqrt((exc[k_] - exc[start_ind])**2 + (inh[k_] - inh[start_ind])**2)
    min_i = start_ind
        
    print(found_solution, already_tried_)
        
    if len(found_solution) == len(already_tried_):
        print("already tried all options")
        min_i = -1
        return min_i
    
    for i_ in found_solution:
        if i_ not in already_tried_:
            if i_ != k_ and i_ != min_i:
                dist_ = np.sqrt((exc[k_] - exc[i_])**2 + (inh[k_] - inh[i_])**2)
                if dist_ < min_dist:
                    min_dist = dist_
                    min_i = i_
                    
    if min_i == 0 and 0 in already_tried_:
        return -1
    
    return min_i

In [4]:
##### LOAD BOUNDARIES
data_file = 'bi.pickle'
with open(data_file,'rb') as f:
    load_array= pickle.load(f)
exc = load_array[0]
inh = load_array[1]
print(len(exc))
#plt.scatter(exc, inh)

147


In [5]:
bestControl_init = [None] * len(exc)
bestState_init = [None] * len(exc)
cost_init = [None] * len(exc)
runtime_init = [None] * len(exc)
grad_init = [None] * len(exc)
phi_init = [None] * len(exc)
costnode_init = [None] * len(exc)
weights_init = [None] * len(exc)

conv_init = [[False]*2] * len(exc)

In [6]:
bestControl_0 = [None] * len(exc)
bestState_0 = [None] * len(exc)
cost_0 = [None] * len(exc)
runtime_0 = [None] * len(exc)
grad_0 = [None] * len(exc)
phi_0 = [None] * len(exc)
costnode_0 = [None] * len(exc)
weights_0 = [None] * len(exc)

conv_0 = [[False]*2] * len(exc)

cgv_list = [None, "HS", "FR", "PR", "CD", "LS", "DY", "WYL", "HZ", None]

In [7]:
initVars = [None] * len(exc)
target = [None] * len(exc)
cost_uncontrolled = [None] * len(exc)

In [8]:
dur_pre = 10
dur_post = 10

n_pre = int(np.around(dur_pre/aln.params.dt + 1.,1))
n_post = int(np.around(dur_post/aln.params.dt + 1.,1))

tol = 1e-32
start_step = 10.
c_scheme = np.zeros(( 1,1 ))
c_scheme[0,0] = 1.
u_mat = np.identity(1)
u_scheme = np.array([[1.]])

c_var = [ [0], [1], [0,1]]
p_var = [ [0], [0], [0]]

### CURRENTS
cntrl_vars_0 = [0,1]
prec_vars = [0]

if case[0] == '0':    # low to high
    max_I = [3., -3.]
elif case[0] == '1':
    max_I = [-3., 3.]
    
if case[1] == '0':    # sparsity
    factor_ws = 1.
    factor_we = 0.
elif case[1] == '1':  # energy
    factor_ws = 0.
    factor_we = 1.
    
if case[3] == '0' and case[2] == '0':
    cntrl_vars_init = [0]
elif case[3] == '1' and case[2] == '0':
    cntrl_vars_init = [1]
elif case[3] == '0' and case[2] == '1':
    cntrl_vars_init = [2,4]
elif case[3] == '1' and case[2] == '1':
    cntrl_vars_init = [3,5]

if case[4] == '0':
    dur = 100
    trans_time = 0.8
elif case[4] == '1':
    dur = 400
    trans_time = 0.95
    
maxC = [5., -5., 0.18, 0.]

n_dur = int(np.around(dur/aln.params.dt + 1.,1))
max_cntrl, min_cntrl = setmaxmincontrol(maxC[0], maxC[1], maxC[2], maxC[3])

In [9]:
init_file = 'control_init_' + case + '.pickle'
final_file = 'control_0_' + case + '.pickle'
final_file_1 = 'control_1_' + case + '.pickle'

In [10]:
if os.path.isfile(init_file) :
    print("file found")
    
    with open(init_file,'rb') as f:
        load_array = pickle.load(f)

    bestControl_init = load_array[0]
    bestState_init = load_array[1]
    cost_init = load_array[2]
    runtime_init = load_array[3]
    grad_init = load_array[4]
    phi_init = load_array[5]
    costnode_init = load_array[6]
    weights_init = load_array[7]

file found


In [11]:
i_stepsize = 11
i_range = range(0, len(exc),i_stepsize)
data.set_parameters(aln)

for i in i_range:
    print("------- ", i, exc[i], inh[i])
    aln.params.mue_ext_mean = exc[i] * 5.
    aln.params.mui_ext_mean = inh[i] * 5.
    
    aln.params.duration = 3000.
    
    control0 = aln.getZeroControl()
    control0 = functions.step_control(aln, maxI_ = max_I[0])

    aln.run(control=control0)
    
    target_rates = np.zeros((2))
    target_rates[0] = aln.rates_exc[0,-1] 
    target_rates[1] = aln.rates_inh[0,-1]

    control0 = functions.step_control(aln, maxI_ = max_I[1])
    aln.run(control=control0)

    init_state_vars = np.zeros(( len(state_vars) ))
    for j in range(len(state_vars)):
        if aln.state[state_vars[j]].size == 1:
            init_state_vars[j] = aln.state[state_vars[j]][0]
        else:
            init_state_vars[j] = aln.state[state_vars[j]][0,-1]

    initVars[i] = init_state_vars
    
    aln.params.duration = dur

    target[i] = aln.getZeroTarget()
    target[i][:,0,:] = target_rates[0]
    target[i][:,1,:] = target_rates[1]

-------  0 0.4000000000000001 0.3500000000000001
-------  11 0.4500000000000001 0.42500000000000016
-------  22 0.5000000000000002 0.4750000000000002
-------  33 0.5000000000000002 0.5250000000000002
-------  44 0.47500000000000014 0.5750000000000003
-------  55 0.4250000000000001 0.6250000000000003
-------  66 0.5250000000000001 0.6500000000000004
-------  77 0.4500000000000001 0.7000000000000004
-------  88 0.5500000000000003 0.7250000000000004
-------  99 0.4250000000000001 0.7750000000000005
-------  110 0.5000000000000002 0.8000000000000005
-------  121 0.5750000000000002 0.8250000000000005
-------  132 0.4500000000000001 0.8750000000000006
-------  143 0.5250000000000001 0.9000000000000006


In [12]:
data.set_parameters(aln)

for i in i_range:
    print("------- ", i, exc[i], inh[i])
    aln.params.mue_ext_mean = exc[i] * 5.
    aln.params.mui_ext_mean = inh[i] * 5.
    
    aln.params.duration = dur
        
    cost.setParams(1.0, 0.0, 0.0)

##### zero control as input for uncontrolled cost
    setinit(initVars[i], aln)
    control0 = aln.getZeroControl()

    # "HS", "FR", "PR", "HZ"
    cgv = None
    max_it = 0

    bestControl_init_, bestState_init_, cost_init_, runtime_init_, grad_init_, phi_init_, costnode_init_ = aln.A1(
        control0, target[i], c_scheme, u_mat, u_scheme, max_iteration_ = max_it, tolerance_ = tol,
        startStep_ = start_step, max_control_ = max_cntrl, min_control_ = min_cntrl, t_sim_ = dur,
        t_sim_pre_ = dur_pre, t_sim_post_ = dur_post, CGVar = cgv, control_variables_ = cntrl_vars_init,
        prec_variables_ = prec_vars, transition_time_ = trans_time)
    
    cost_uncontrolled[i] = cost_init_[0]

-------  0 0.4000000000000001 0.3500000000000001
set cost params:  1.0 0.0 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  5902.406479238383
Gradient descend method:  None
RUN  0 , total integrated cost =  5902.406479238383
Improved over  0  iterations in  0.0  seconds by  0.0  percent.
-------  11 0.4500000000000001 0.42500000000000016
set cost params:  1.0 0.0 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  13311.435770934964
Gradient descend method:  None
RUN  0 , total integrated cost =  13311.435770934964
Improved over  0  iterations in  0.0  seconds by  0.0  percent.
-------  22 0.5000000000000002 0.4750000000000002
set cost params:  1.0 0.0 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  21565.177588309605
Gradient descend method:  None
RUN  0 , total integrated cost =  21565.177588309605
Improved over  0  iterations in  0.0  seconds by  0.0  percent.
-------  33 0.5000000000000002 0.525000000000

In [13]:
i_range_ = []

for i in i_range:
    if type(bestControl_init[i]) == type(None):
        i_range_.append(i)

i_range = np.array(i_range_)
        
print(i_range)

[ 11  22  33  44  55  66  88  99 110 121 132 143]


In [14]:
factor_iteration = 20.

for i in i_range:
    print("------- ", i, exc[i], inh[i])
    aln.params.mue_ext_mean = exc[i] * 5.
    aln.params.mui_ext_mean = inh[i] * 5.
    
    aln.params.duration = dur
        
##### zero control as input for uncontrolled cost
    setinit(initVars[i], aln)
    control0 = aln.getZeroControl()

##### initial guess
    weight_ = 10
    cost.setParams(1.0, weight_ * factor_we, weight_ * factor_ws)

    setinit(initVars[i], aln)
    #if prev_i != -1:
    #    control0 = bestControl_init[prev_i][:,:,n_pre-1:-n_post+1]
    
    # "HS", "FR", "PR", "HZ"
    cgv = None
    max_it = int(100 * factor_iteration)
    
    weights_init[i] = cost.getParams()

    bestControl_init[i], bestState_init[i], cost_init[i], runtime_init[i], grad_init[i], phi_init[i], costnode_init[i] = aln.A1(
        control0, target[i], c_scheme, u_mat, u_scheme, max_iteration_ = max_it, tolerance_ = tol,
        startStep_ = start_step, max_control_ = max_cntrl, min_control_ = min_cntrl, t_sim_ = dur,
        t_sim_pre_ = dur_pre, t_sim_post_ = dur_post, CGVar = cgv, control_variables_ = cntrl_vars_init,
        prec_variables_ = prec_vars, transition_time_ = trans_time)
    
    j = 1
    while cost_init[i][-j] == 0.:
        j += 1
    
    weight_ = 10 * cost_uncontrolled[i] / cost_init[i][-j]
    print("weight = ", weight_)
    cost.setParams(1.0, weight_ * factor_we, weight_ * factor_ws)

    setinit(initVars[i], aln)
    control0 = bestControl_init[i][:,:,n_pre-1:-n_post+1]

    # "HS", "FR", "PR", "HZ"
    cgv = None
    max_it = int(500 * factor_iteration)

    weights_init[i] = cost.getParams()
    
    bestControl_init[i], bestState_init[i], cost_init[i], runtime_init[i], grad_init[i], phi_init[i], costnode_init[i] = aln.A1(
        control0, target[i], c_scheme, u_mat, u_scheme, max_iteration_ = max_it, tolerance_ = tol,
        startStep_ = start_step, max_control_ = max_cntrl, min_control_ = min_cntrl, t_sim_ = dur,
        t_sim_pre_ = dur_pre, t_sim_post_ = dur_post, CGVar = cgv, control_variables_ = cntrl_vars_init,
        prec_variables_ = prec_vars, transition_time_ = trans_time)
    
    prev_i = i

-------  11 0.4500000000000001 0.42500000000000016
set cost params:  1.0 0.0 10.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  13311.435770934964
Gradient descend method:  None
RUN  1 , total integrated cost =  2573.7692663690545
RUN  2 , total integrated cost =  1298.668177334914
RUN  3 , total integrated cost =  918.5613980336879
RUN  4 , total integrated cost =  601.7459233896169
RUN  5 , total integrated cost =  454.31531571322216
RUN  6 , total integrated cost =  290.76011639810724
RUN  7 , total integrated cost =  217.63140093749277
RUN  8 , total integrated cost =  114.58019145203934
RUN  9 , total integrated cost =  88.1595293099059
RUN  10 , total integrated cost =  70.2253004461597
RUN  11 , total integrated cost =  60.56452817011697
RUN  12 , total integrated cost =  56.913046335034096
RUN  13 , total integrated cost =  54.21870803986968
RUN  14 , total integrated cost =  50.397311109995826
RUN  15 , total integrated cost =  47.863840701798495
RUN 

ERROR:root:Problem in initial value trasfer


Control only changes marginally.
RUN  1244 , total integrated cost =  22.748403185257846
Improved over  1244  iterations in  36.12323762848973  seconds by  99.8291063144749  percent.
Problem in initial value trasfer:  Vmean_exc -56.672352757090884 -56.672352771956916
weight =  5851.591279849247
set cost params:  1.0 0.0 5851.591279849247
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  13307.916504934521
Gradient descend method:  None
RUN  1 , total integrated cost =  13140.408042081435
RUN  2 , total integrated cost =  13140.149846110337
RUN  3 , total integrated cost =  13139.773534278193
RUN  4 , total integrated cost =  13080.174703719957
RUN  5 , total integrated cost =  13024.329392680718
RUN  6 , total integrated cost =  13024.127646718904
RUN  7 , total integrated cost =  13024.122351598447
RUN  8 , total integrated cost =  13024.121851005671


ERROR:root:Problem in initial value trasfer


RUN  9 , total integrated cost =  13024.12181596667
RUN  10 , total integrated cost =  13024.121814685523
RUN  11 , total integrated cost =  13024.12181465512
RUN  12 , total integrated cost =  13024.121814654512
RUN  13 , total integrated cost =  13024.121814654462
RUN  14 , total integrated cost =  13024.121814654462
Control only changes marginally.
RUN  14 , total integrated cost =  13024.121814654462
Improved over  14  iterations in  0.3379777353256941  seconds by  2.1325253293769038  percent.
Problem in initial value trasfer:  Vmean_exc -56.67242208098633 -56.67241812564137
-------  22 0.5000000000000002 0.4750000000000002
set cost params:  1.0 0.0 10.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  21565.177588309605
Gradient descend method:  None
RUN  1 , total integrated cost =  2385.6185871031903
RUN  2 , total integrated cost =  191.43912591037028
RUN  3 , total integrated cost =  174.239929937454
RUN  4 , total integrated cost =  160.97732207157094
R

...

Control only changes marginally.
RUN  29 , total integrated cost =  24265.194910917617
Improved over  29  iterations in  0.6723793726414442  seconds by  1.9132428163843969  percent.
Problem in initial value trasfer:  Vmean_exc -56.702099847102154 -56.70209955798772
-------  88 0.5500000000000003 0.7250000000000004
set cost params:  1.0 0.0 10.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  29125.79663583254
Gradient descend method:  None
RUN  1 , total integrated cost =  4635.670702018207
RUN  2 , total integrated cost =  380.5119419124945
RUN  3 , total integrated cost =  343.0408883991968
RUN  4 , total integrated cost =  316.13583755856666
RUN  5 , total integrated cost =  289.039107193469
RUN  6 , total integrated cost =  269.02550092795786
RUN  7 , total integrated cost =  247.67918478403635
RUN  8 , total integrated cost =  231.74102217406062
RUN  9 , total integrated cost =  214.10356335744186
RUN  10 , total integrated cost =  200.38943480118354
RUN  1

ERROR:root:Problem in initial value trasfer


Control only changes marginally.
RUN  39 , total integrated cost =  28816.94716578898
Improved over  39  iterations in  0.8859561197459698  seconds by  1.044811818035555  percent.
Problem in initial value trasfer:  Vmean_exc -56.70419356992839 -56.70419358192549
-------  99 0.4250000000000001 0.7750000000000005
set cost params:  1.0 0.0 10.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  6120.770160604903
Gradient descend method:  None
RUN  1 , total integrated cost =  6120.770160604903
Control only changes marginally.
RUN  1 , total integrated cost =  6120.770160604903
Improved over  1  iterations in  0.04441416822373867  seconds by  0.0  percent.
weight =  10.0
set cost params:  1.0 0.0 10.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  6120.770160604903
Gradient descend method:  None
RUN  1 , total integrated cost =  6120.770160604903
Control only changes marginally.
RUN  1 , total integrated cost =  6120.770160604903
Improved over 

ERROR:root:Problem in initial value trasfer


RUN  13 , total integrated cost =  33132.77759528316
RUN  14 , total integrated cost =  33132.77759528316
Control only changes marginally.
RUN  14 , total integrated cost =  33132.77759528316
Improved over  14  iterations in  0.3712823763489723  seconds by  1.4708864149403666  percent.
Problem in initial value trasfer:  Vmean_exc -56.7034358072032 -56.7034357533006
-------  132 0.4500000000000001 0.8750000000000006
set cost params:  1.0 0.0 10.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  10117.573589110178
Gradient descend method:  None
RUN  1 , total integrated cost =  10117.573589110178
Control only changes marginally.
RUN  1 , total integrated cost =  10117.573589110178
Improved over  1  iterations in  0.04942931979894638  seconds by  0.0  percent.
weight =  10.0
set cost params:  1.0 0.0 10.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  10117.573589110178
Gradient descend method:  None
RUN  1 , total integrated cost =  10117.5

In [15]:
#plot initial guesses
"""
for i in i_range:
        
    aln.params.mue_ext_mean = exc[i] * 5.
    aln.params.mui_ext_mean = inh[i] * 5.

    plotFunc.plot_control_current(aln, [bestControl_init[i]],
        [costnode_init[i]], [weights_init[i]], dur,
        dur_pre, dur_post, initVars[i], target[i], '', filename_ = '', transition_time_ = trans_time,
        labels_ = ["init", "sparse control" + str(i)], print_cost_ = False)
"""

'\nfor i in i_range:\n        \n    aln.params.mue_ext_mean = exc[i] * 5.\n    aln.params.mui_ext_mean = inh[i] * 5.\n\n    plotFunc.plot_control_current(aln, [bestControl_init[i]],\n        [costnode_init[i]], [weights_init[i]], dur,\n        dur_pre, dur_post, initVars[i], target[i], \'\', filename_ = \'\', transition_time_ = trans_time,\n        labels_ = ["init", "sparse control" + str(i)], print_cost_ = False)\n'

In [16]:
found_solution = []
no_solution = []
last_update = -1
factor_iteration = 20.
already_tried = [ [] for _ in range(len(exc)) ]
i_range = range(0, len(exc),i_stepsize)
i_range_0 = []
i_range_1 = []

print(already_tried, len(already_tried))

for k in range(len(i_range)**2):
    print('------------------------------------------------------------')
    print('--------------------', k)
    print('------------------------------------------------------------')
        
    print("found solution: ", found_solution)
    print("no solution: ", no_solution)
    
    if len(i_range) == len(found_solution) + len(no_solution):
        print("found solution for all parameters")
        break
    
    #if last_update != k-1:
    #    print("no improvement from previous step")
    #    break

    for i in i_range:
        print("------- ", i, exc[i], inh[i])

        if np.abs(bestState_init[i][0,0,-1] - target[i][0,0,-1]) < 0.5 * np.abs(
            bestState_init[i][0,0,-1] - bestState_init[i][0,0,0]):
            if i not in found_solution:
                print("found solution for ", i)
                found_solution.append(i)
                #last_update = k
            if i in no_solution:
                no_solution.pop(no_solution.index(i))
            continue

        #if i not in no_solution:
        #    print("no solution for ", i)
        #    no_solution.append(i)
        
        if i not in i_range_0:
            i_range_0.append(i)
            
        if i not in i_range_1:
            i_range_1.append(i)

        closest_ = getclosest(i, found_solution, exc, inh, already_tried[i])
        print("closest index ", closest_)

        weight_ = 10
        cost.setParams(1.0, weight_ * factor_we, weight_ * factor_ws)

        setinit(initVars[i], aln)
        aln.params.mue_ext_mean = exc[i] * 5.
        aln.params.mui_ext_mean = inh[i] * 5.
            
        if i != 0 and closest_ != -1:
            control0 = bestControl_init[closest_][:,:,n_pre-1:-n_post+1]
            if closest_ not in already_tried[i]:
                already_tried[i].append(closest_)
                        
        if closest_ == -1:
            print("all options tried already")
            if i not in no_solution:
                no_solution.append(i)
                continue

        # "HS", "FR", "PR", "HZ"
        cgv = None
        max_it = int(100 * factor_iteration)

        weights_init[i] = cost.getParams()
        
        print("precision vars = ", prec_vars)

        bestControl_init[i], bestState_init[i], cost_init[i], runtime_init[i], grad_init[i], phi_init[i], costnode_init[i] = aln.A1(
            control0, target[i], c_scheme, u_mat, u_scheme, max_iteration_ = max_it, tolerance_ = tol,
            startStep_ = start_step, max_control_ = max_cntrl, min_control_ = min_cntrl, t_sim_ = dur,
            t_sim_pre_ = dur_pre, t_sim_post_ = dur_post, CGVar = cgv, control_variables_ = cntrl_vars_init,
            prec_variables_ = prec_vars, transition_time_ = trans_time)

        j = 1
        while cost_init[i][-j] == 0.:
            j += 1

        weight_ = 10 * cost_uncontrolled[i] / cost_init[i][-j]
        print("weight = ", weight_)
        cost.setParams(1.0, weight_ * factor_we, weight_ * factor_ws)

        setinit(initVars[i], aln)
        control0 = bestControl_init[i][:,:,n_pre-1:-n_post+1]

        # "HS", "FR", "PR", "HZ"
        cgv = None
        max_it = int(500 * factor_iteration)

        weights_init[i] = cost.getParams()

        bestControl_init[i], bestState_init[i], cost_init[i], runtime_init[i], grad_init[i], phi_init[i], costnode_init[i] = aln.A1(
            control0, target[i], c_scheme, u_mat, u_scheme, max_iteration_ = max_it, tolerance_ = tol,
            startStep_ = start_step, max_control_ = max_cntrl, min_control_ = min_cntrl, t_sim_ = dur,
            t_sim_pre_ = dur_pre, t_sim_post_ = dur_post, CGVar = cgv, control_variables_ = cntrl_vars_init,
            prec_variables_ = prec_vars, transition_time_ = trans_time)

[[], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], []] 147
------------------------------------------------------------
-------------------- 0
------------------------------------------------------------
found solution:  []
no solution:  []
-------  0 0.4000000000000001 0.3500000000000001
found solution for  0
-------  11 0.4500000000000001 0.42500000000000016
found solution for  11
-------  22 0.5000000000000002 0.4750000000000002
found solution for  22
-------

ERROR:root:Problem in initial value trasfer


Control only changes marginally.
RUN  677 , total integrated cost =  74.38329042250555
Improved over  677  iterations in  13.62904241681099  seconds by  98.95865063820732  percent.
Problem in initial value trasfer:  Vmean_exc -56.631599827555966 -56.63159981075854
weight =  956.2515072336719
set cost params:  1.0 0.0 956.2515072336719
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  7112.472251523473
Gradient descend method:  None
RUN  1 , total integrated cost =  7108.1345706493175
RUN  2 , total integrated cost =  7108.090603846126
RUN  3 , total integrated cost =  7108.067611082594
RUN  4 , total integrated cost =  7107.247505267654
RUN  5 , total integrated cost =  7105.9553794506455
RUN  6 , total integrated cost =  7105.925966910164
RUN  7 , total integrated cost =  7105.920787881256
RUN  8 , total integrated cost =  7105.91466194434
RUN  9 , total integrated cost =  7105.85098746864
RUN  10 , total integrated cost =  7105.345335821375
RUN  11 , total integ

ERROR:root:Problem in initial value trasfer


RUN  60 , total integrated cost =  7102.037498682619
Control only changes marginally.
RUN  62 , total integrated cost =  7102.037498682618
Improved over  62  iterations in  1.2825526762753725  seconds by  0.1467106298884886  percent.
Problem in initial value trasfer:  Vmean_exc -56.63168647131968 -56.63168348707226
-------  66 0.5250000000000001 0.6500000000000004
found solution for  66
-------  77 0.4500000000000001 0.7000000000000004
found solution for  77
-------  88 0.5500000000000003 0.7250000000000004
found solution for  88
-------  99 0.4250000000000001 0.7750000000000005
[0, 11, 22, 33, 66, 77, 88] []
closest index  77
set cost params:  1.0 0.0 10.0
precision vars =  [0]
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  6192.015767175609
Gradient descend method:  None
RUN  1 , total integrated cost =  6123.265788356299
RUN  2 , total integrated cost =  6120.813595349729
RUN  3 , total integrated cost =  6120.770870740274
RUN  4 , total integrated cost =  6

...

RUN  80 , total integrated cost =  6115.999114444091
Control only changes marginally.
RUN  84 , total integrated cost =  6115.999114444073
Improved over  84  iterations in  1.7572296056896448  seconds by  0.07390366663173609  percent.
Problem in initial value trasfer:  Vmean_exc -56.62557447781285 -56.62557272579751
-------  110 0.5000000000000002 0.8000000000000005
found solution for  110
-------  121 0.5750000000000002 0.8250000000000005
-------  132 0.4500000000000001 0.8750000000000006
found solution for  132
-------  143 0.5250000000000001 0.9000000000000006
found solution for  143
------------------------------------------------------------
-------------------- 2
------------------------------------------------------------
found solution:  [0, 11, 22, 33, 66, 77, 88, 121, 55, 110, 132, 143]
no solution:  []
-------  0 0.4000000000000001 0.3500000000000001
-------  11 0.4500000000000001 0.42500000000000016
-------  22 0.5000000000000002 0.4750000000000002
-------  33 0.50000000000

ERROR:root:Problem in initial value trasfer


Control only changes marginally.
RUN  367 , total integrated cost =  48.77558721080246
Improved over  367  iterations in  7.536675246432424  seconds by  60.891058887598135  percent.
Problem in initial value trasfer:  Vmean_exc -56.68406098822788 -56.68406103657918
weight =  3308.421366639967
set cost params:  1.0 0.0 3308.421366639967
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  16133.831184880337
Gradient descend method:  None
RUN  1 , total integrated cost =  16065.824837121307
RUN  2 , total integrated cost =  16065.697153380655
RUN  3 , total integrated cost =  16065.695952830703
RUN  4 , total integrated cost =  16065.695950251817
RUN  5 , total integrated cost =  16065.695950072968
RUN  6 , total integrated cost =  16065.69595006062
RUN  7 , total integrated cost =  16065.69595005995
RUN  8 , total integrated cost =  16065.695950059926


ERROR:root:Problem in initial value trasfer


RUN  9 , total integrated cost =  16065.69595005992
RUN  10 , total integrated cost =  16065.695950059913
RUN  11 , total integrated cost =  16065.695950059911
RUN  12 , total integrated cost =  16065.695950059911
Control only changes marginally.
RUN  12 , total integrated cost =  16065.695950059911
Improved over  12  iterations in  0.2904694117605686  seconds by  0.4223128036958599  percent.
Problem in initial value trasfer:  Vmean_exc -56.68396977599029 -56.68397229407174
-------  55 0.4250000000000001 0.6250000000000003
-------  66 0.5250000000000001 0.6500000000000004
-------  77 0.4500000000000001 0.7000000000000004
-------  88 0.5500000000000003 0.7250000000000004
-------  99 0.4250000000000001 0.7750000000000005
found solution for  99
-------  110 0.5000000000000002 0.8000000000000005
-------  121 0.5750000000000002 0.8250000000000005
-------  132 0.4500000000000001 0.8750000000000006
-------  143 0.5250000000000001 0.9000000000000006
--------------------------------------------

In [17]:
factor_iteration = 20
full_converge = False
conv_init = [[False]*2] * len(exc)

for i in range(len(conv_init)):
    if i not in i_range:
        conv_init[i] = [True, True]
        
counter = 0

while full_converge == False:
    
    if counter > 100:
        break
        
    print(conv_init[::i_stepsize])
    full_converge = True
    
    for conv in conv_init[::i_stepsize]:
        if not conv[0]:
            full_converge = False
            break
        if not conv[1]:
            full_converge = False
            break
    
    if full_converge:
        print("full convergence")
        break

    for i in i_range:        

        print("------- ", i, exc[i], inh[i])
        aln.params.mue_ext_mean = exc[i] * 5.
        aln.params.mui_ext_mean = inh[i] * 5.
        
        j = 1
        while cost_init[i][-j] == 0.:
            j += 1
            
        if j == cost_init[i].shape[0]-1:
            print("converged for ", i)
            if conv_init[i][0]:
                conv_init[i] = [True, True]
            else:
                conv_init[i] = [True, False]
            continue
    
        print("no convergence")
            
        weight_ = (factor_we * weights_init[i][1] * cost_uncontrolled[i] / cost_init[i][-j]
                   + factor_ws * weights_init[i][2] * cost_uncontrolled[i] / cost_init[i][-j]) - 1
        print("weight = ", weight_)
        cost.setParams(1.0, weight_ * factor_we, weight_ * factor_ws)

        setinit(initVars[i], aln)
        control0 = bestControl_init[i][:,:,n_pre-1:-n_post+1]

        # "HS", "FR", "PR", "HZ"
        cgv = None
        max_it = int( 500 * factor_iteration )

        weights_init[i] = cost.getParams()

        bestControl_init[i], bestState_init[i], cost_init[i], runtime_init[i], grad_init[i], phi_init[i], costnode_init[i] = aln.A1(
            control0, target[i], c_scheme, u_mat, u_scheme, max_iteration_ = max_it, tolerance_ = tol,
            startStep_ = start_step, max_control_ = max_cntrl, min_control_ = min_cntrl, t_sim_ = dur,
            t_sim_pre_ = dur_pre, t_sim_post_ = dur_post, CGVar = cgv, control_variables_ = cntrl_vars_init,
            prec_variables_ = prec_vars, transition_time_ = trans_time)
        
        counter += 1

[[False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False]]
-------  0 0.4000000000000001 0.3500000000000001
converged for  0
-------  11 0.4500000000000001 0.42500000000000016
no convergence
weight =  5979.678205253944
set cost params:  1.0 0.0 5979.678205253944
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  13306.993508473057
Gradient descend method:  None
RUN  1 , total integrated cost =  13306.97693453539
RUN  2 , total integrated cost =  13306.976485069574
RUN  3 , total integrated cost =  13306.976456526429
RUN  4 , total integrated cost =  13306.9764550469
RUN  5 , total integrated cost =  13306.976454956333
RUN  6 , total integrated cost =  13306.976454952322
RUN  7 , total integrated cost =  13306.976454952204


ERROR:root:Problem in initial value trasfer


RUN  8 , total integrated cost =  13306.976454952173
RUN  9 , total integrated cost =  13306.976454952168
RUN  10 , total integrated cost =  13306.976454952168
Control only changes marginally.
RUN  10 , total integrated cost =  13306.976454952168
Improved over  10  iterations in  0.2795524653047323  seconds by  0.00012815457434101063  percent.
Problem in initial value trasfer:  Vmean_exc -56.672416925576044 -56.67241309940236
-------  22 0.5000000000000002 0.4750000000000002
no convergence
weight =  12374.275462279224
set cost params:  1.0 0.0 12374.275462279224
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  21560.133132665276
Gradient descend method:  None
RUN  1 , total integrated cost =  21560.096733311722
RUN  2 , total integrated cost =  21560.095628200826
RUN  3 , total integrated cost =  21560.095615292936


ERROR:root:Problem in initial value trasfer


RUN  4 , total integrated cost =  21560.095614890426
RUN  5 , total integrated cost =  21560.0956148707
RUN  6 , total integrated cost =  21560.095614870064
RUN  7 , total integrated cost =  21560.095614870057
RUN  8 , total integrated cost =  21560.09561487004
RUN  9 , total integrated cost =  21560.09561487004
Control only changes marginally.
RUN  9 , total integrated cost =  21560.09561487004
Improved over  9  iterations in  0.2627545651048422  seconds by  0.00017401467331978893  percent.
Problem in initial value trasfer:  Vmean_exc -56.69838310098527 -56.698383133112294
-------  33 0.5000000000000002 0.5250000000000002
no convergence
weight =  6969.751907739501
set cost params:  1.0 0.0 6969.751907739501
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  21064.287576593404
Gradient descend method:  None
RUN  1 , total integrated cost =  21064.281209773824
RUN  2 , total integrated cost =  21064.28090259592
RUN  3 , total integrated cost =  21064.28087043358
RUN

...

RUN  1 , total integrated cost =  24738.40473155185
RUN  2 , total integrated cost =  24738.40473155185
Control only changes marginally.
RUN  2 , total integrated cost =  24738.40473155185
Improved over  2  iterations in  0.11638337559998035  seconds by  2.842170943040401e-14  percent.
Problem in initial value trasfer:  Vmean_exc -56.702099745921124 -56.70209946066033
-------  77 0.4500000000000001 0.7000000000000004
converged for  77
-------  88 0.5500000000000003 0.7250000000000004
converged for  88
-------  99 0.4250000000000001 0.7750000000000005
converged for  99
-------  110 0.5000000000000002 0.8000000000000005
converged for  110
-------  121 0.5750000000000002 0.8250000000000005
no convergence
weight =  13903.837118622285
set cost params:  1.0 0.0 13903.837118622285
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  33630.039295621864
Gradient descend method:  None
RUN  1 , total integrated cost =  33630.039295621864
Control only changes marginally.
RUN  1 

ERROR:root:Problem in initial value trasfer
ERROR:root:Problem in initial value trasfer
ERROR:root:Problem in initial value trasfer


RUN  1 , total integrated cost =  23426.003800633644
Control only changes marginally.
RUN  1 , total integrated cost =  23426.003800633644
Improved over  1  iterations in  0.07409851253032684  seconds by  0.0  percent.
Problem in initial value trasfer:  Vmean_exc -56.70054162743189 -56.70054168440823
[[True, True], [False, False], [False, False], [False, False], [False, False], [True, False], [False, False], [True, True], [True, False], [True, True], [True, True], [False, False], [True, False], [False, False]]
-------  0 0.4000000000000001 0.3500000000000001
converged for  0
-------  11 0.4500000000000001 0.42500000000000016
converged for  11
-------  22 0.5000000000000002 0.4750000000000002
no convergence
weight =  12376.20503783867
set cost params:  1.0 0.0 12376.20503783867
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  21563.435257176097
Gradient descend method:  None
RUN  1 , total integrated cost =  21563.435257176097
Control only changes marginally.
RUN 

ERROR:root:Problem in initial value trasfer


RUN  0 , total integrated cost =  24738.40479761799
Gradient descend method:  None
RUN  1 , total integrated cost =  24738.40479761799
Control only changes marginally.
RUN  1 , total integrated cost =  24738.40479761799
Improved over  1  iterations in  0.07179366424679756  seconds by  0.0  percent.
Problem in initial value trasfer:  Vmean_exc -56.702099745921124 -56.70209946066033
-------  77 0.4500000000000001 0.7000000000000004
converged for  77
-------  88 0.5500000000000003 0.7250000000000004
converged for  88
-------  99 0.4250000000000001 0.7750000000000005
converged for  99
-------  110 0.5000000000000002 0.8000000000000005
converged for  110
-------  121 0.5750000000000002 0.8250000000000005
converged for  121
-------  132 0.4500000000000001 0.8750000000000006
converged for  132
-------  143 0.5250000000000001 0.9000000000000006
converged for  143
[[True, True], [True, False], [False, False], [False, False], [True, False], [True, True], [False, False], [True, True], [True, True

In [18]:
print(conv_init[::i_stepsize])

with open(init_file,'wb') as f:
    pickle.dump([bestControl_init, bestState_init, cost_init, runtime_init, grad_init, phi_init,
                 costnode_init, weights_init], f)

[[True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True]]


In [19]:
if os.path.isfile(final_file) :
    print("file found")
    
    with open(final_file,'rb') as f:
        load_array = pickle.load(f)

    bestControl_0 = load_array[0]
    bestState_0 = load_array[1]
    cost_0 = load_array[2]
    runtime_0 = load_array[3]
    grad_0 = load_array[4]
    phi_0 = load_array[5]
    costnode_0 = load_array[6]
    weights_0 = load_array[7]

file found


In [20]:
i_range_0 = range(0, len(exc),i_stepsize)
i_range_ = []

for i in i_range_0:
    if type(bestControl_0[i]) == type(None):
        i_range_.append(i)

i_range_0 = np.array(i_range_)
        
print(i_range_0)

[ 11  22  33  44  55  66  88  99 110 121 132 143]


In [21]:
factor_iteration = 20
    
for i in i_range_0:
    print("------- ", i, exc[i], inh[i])
    aln.params.mue_ext_mean = exc[i] * 5.
    aln.params.mui_ext_mean = inh[i] * 5.

# exc and inh control current 

    setinit(initVars[i], aln)
    aln.params.duration = dur
    control0 = bestControl_init[i][:,:,n_pre-1:-n_post+1].copy()
    #control0 = bestControl_0[i][:,:,n_pre-1:-n_post+1]
 
    cost.setParams(1.0, 0.0, 10.0)

    # "HS", "FR", "PR", "HZ"
    cgv = None
    max_it = 50 * factor_iteration

    weights_0[i] = cost.getParams()

    bestControl_0[i], bestState_0[i], cost_0[i], runtime_0[i], grad_0[i], phi_0[i], costnode_0[i] = aln.A1(
        control0, target[i], c_scheme, u_mat, u_scheme, max_iteration_ = max_it, tolerance_ = tol,
        startStep_ = start_step, max_control_ = max_cntrl, min_control_ = min_cntrl, t_sim_ = dur,
        t_sim_pre_ = dur_pre, t_sim_post_ = dur_post, CGVar = cgv, control_variables_ = cntrl_vars_0,
        prec_variables_ = prec_vars, transition_time_ = trans_time)
    
    j = 1
    while cost_0[i][-j] == 0.:
        j += 1
    
    weight_ = 100
    print("weight = ", weight_)
    cost.setParams(1.0, weight_ * factor_we, weight_ * factor_ws)

    setinit(initVars[i], aln)
    control0 = bestControl_0[i][:,:,n_pre-1:-n_post+1]

    # "HS", "FR", "PR", "HZ"
    cgv = "HS"
    max_it = 50 * factor_iteration

    weights_0[i] = cost.getParams()

    bestControl_0[i], bestState_0[i], cost_0[i], runtime_0[i], grad_0[i], phi_0[i], costnode_0[i] = aln.A1(
        control0, target[i], c_scheme, u_mat, u_scheme, max_iteration_ = max_it, tolerance_ = tol,
        startStep_ = start_step, max_control_ = max_cntrl, min_control_ = min_cntrl, t_sim_ = dur,
        t_sim_pre_ = dur_pre, t_sim_post_ = dur_post, CGVar = cgv, control_variables_ = cntrl_vars_0,
        prec_variables_ = prec_vars, transition_time_ = trans_time)
    
    j = 1
    while cost_0[i][-j] == 0.:
        j += 1
    
    weight_ = 100 * cost_uncontrolled[i] / cost_0[i][-j] - 1
    print("weight = ", weight_)
    cost.setParams(1.0, weight_ * factor_we, weight_ * factor_ws)

    setinit(initVars[i], aln)
    control0 = bestControl_0[i][:,:,n_pre-1:-n_post+1]

    # "HS", "FR", "PR", "HZ"
    cgv = None
    max_it = 50 * factor_iteration

    weights_0[i] = cost.getParams()

    bestControl_0[i], bestState_0[i], cost_0[i], runtime_0[i], grad_0[i], phi_0[i], costnode_0[i] = aln.A1(
        control0, target[i], c_scheme, u_mat, u_scheme, max_iteration_ = max_it, tolerance_ = tol,
        startStep_ = start_step, max_control_ = max_cntrl, min_control_ = min_cntrl, t_sim_ = dur,
        t_sim_pre_ = dur_pre, t_sim_post_ = dur_post, CGVar = cgv, control_variables_ = cntrl_vars_0,
        prec_variables_ = prec_vars, transition_time_ = trans_time)

-------  11 0.4500000000000001 0.42500000000000016
set cost params:  1.0 0.0 10.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  125.742633422307
Gradient descend method:  None
RUN  1 , total integrated cost =  25.494722023042293
RUN  2 , total integrated cost =  24.687791759074653
RUN  3 , total integrated cost =  24.13852728546646
RUN  4 , total integrated cost =  23.892925437515483
RUN  5 , total integrated cost =  23.66695617629635
RUN  6 , total integrated cost =  23.52846420336392
RUN  7 , total integrated cost =  23.381341448714572
RUN  8 , total integrated cost =  23.28854110922139
RUN  9 , total integrated cost =  23.181120234571594
RUN  10 , total integrated cost =  23.113904849839027
RUN  11 , total integrated cost =  22.997715692827562
RUN  12 , total integrated cost =  22.93032983043572
RUN  13 , total integrated cost =  22.795075205043794
RUN  14 , total integrated cost =  22.76475042781299
RUN  15 , total integrated cost =  22.72925058953153
RUN 

ERROR:root:Problem in initial value trasfer


Control only changes marginally.
RUN  174 , total integrated cost =  22.493703442792572
Improved over  174  iterations in  11.992560623213649  seconds by  82.11131512790304  percent.
Problem in initial value trasfer:  Vmean_exc -56.6723490086341 -56.672349200602426
weight =  100
set cost params:  1.0 0.0 100.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  224.86532948390519
Gradient descend method:  HS
RUN  1 , total integrated cost =  224.76322051375894
RUN  2 , total integrated cost =  224.58972839560357
RUN  3 , total integrated cost =  224.58857464199792
RUN  4 , total integrated cost =  224.49006845799494
RUN  5 , total integrated cost =  224.4900684579945


ERROR:root:Problem in initial value trasfer


RUN  6 , total integrated cost =  224.49006845799448
RUN  7 , total integrated cost =  224.49006845799448
Control only changes marginally.
RUN  7 , total integrated cost =  224.49006845799448
Improved over  7  iterations in  0.6821299102157354  seconds by  0.16688256334222729  percent.
Problem in initial value trasfer:  Vmean_exc -56.672285235644615 -56.672287792691975
weight =  5928.632371877395
set cost params:  1.0 0.0 5928.632371877395
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  13298.125279870994
Gradient descend method:  None
RUN  1 , total integrated cost =  13160.401030020892
RUN  2 , total integrated cost =  13160.263630403559
RUN  3 , total integrated cost =  13160.242874184023
RUN  4 , total integrated cost =  13160.231641225208
RUN  5 , total integrated cost =  13160.219154883112
RUN  6 , total integrated cost =  13160.203325430568
RUN  7 , total integrated cost =  13160.144678757726
RUN  8 , total integrated cost =  13159.085923481602
RUN  9 , t

...

Control only changes marginally.
RUN  122 , total integrated cost =  47.73446277111619
Improved over  122  iterations in  13.113054970279336  seconds by  34.27817373861137  percent.
Problem in initial value trasfer:  Vmean_exc -56.70054678920583 -56.70054667715045
weight =  100
set cost params:  1.0 0.0 100.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  477.3054934320885
Gradient descend method:  HS
RUN  1 , total integrated cost =  477.27740982044986
RUN  2 , total integrated cost =  476.8075980290653
RUN  3 , total integrated cost =  476.7448007434239
RUN  4 , total integrated cost =  476.0105173364935
RUN  5 , total integrated cost =  475.99638621369553
RUN  6 , total integrated cost =  475.93255585749097
RUN  7 , total integrated cost =  475.9319867256734
RUN  8 , total integrated cost =  475.65147676059007
RUN  9 , total integrated cost =  475.6514642564054
RUN  10 , total integrated cost =  475.63535177407624


ERROR:root:Problem in initial value trasfer


RUN  11 , total integrated cost =  475.63535177407596
RUN  12 , total integrated cost =  475.63535177407596
Control only changes marginally.
RUN  12 , total integrated cost =  475.63535177407596
Improved over  12  iterations in  1.5143098384141922  seconds by  0.34991042026423713  percent.
Problem in initial value trasfer:  Vmean_exc -56.70056130803524 -56.700560347003716
weight =  4925.204234243626
set cost params:  1.0 0.0 4925.204234243626
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  23409.92224025423
Gradient descend method:  None
RUN  1 , total integrated cost =  23381.403314490115
RUN  2 , total integrated cost =  23381.25483421766


ERROR:root:Problem in initial value trasfer


RUN  3 , total integrated cost =  23381.25483421765
RUN  4 , total integrated cost =  23381.25483421765
Control only changes marginally.
RUN  4 , total integrated cost =  23381.25483421765
Improved over  4  iterations in  0.5977649837732315  seconds by  0.12245835651383175  percent.
Problem in initial value trasfer:  Vmean_exc -56.70055576771882 -56.700555001162165


In [22]:
"""
for i in i_range_0:
    
    aln.params.mue_ext_mean = exc[i] * 5.
    aln.params.mui_ext_mean = inh[i] * 5.

    plotFunc.plot_control_current(aln, [bestControl_init[i], bestControl_0[i]],
        [costnode_init[i], costnode_0[i]], [weights_init[i], weights_0[i]], dur,
        dur_pre, dur_post, initVars[i], target[i], '', filename_ = '', transition_time_ = trans_time,
        labels_ = ["init", "sparse control" + str(i)], print_cost_ = False)
"""

'\nfor i in i_range_0:\n    \n    aln.params.mue_ext_mean = exc[i] * 5.\n    aln.params.mui_ext_mean = inh[i] * 5.\n\n    plotFunc.plot_control_current(aln, [bestControl_init[i], bestControl_0[i]],\n        [costnode_init[i], costnode_0[i]], [weights_init[i], weights_0[i]], dur,\n        dur_pre, dur_post, initVars[i], target[i], \'\', filename_ = \'\', transition_time_ = trans_time,\n        labels_ = ["init", "sparse control" + str(i)], print_cost_ = False)\n'

In [23]:
factor_iteration = 20
full_converge = False
conv_0 = [[False]*2] * len(exc)

for i in range(len(conv_0)):
    if i not in i_range_0:
        conv_0[i] = [True, True]

counter = 0

while full_converge == False:
    print('---------------', counter)
    
    if counter > 100:
        break
    
    print(conv_0[::i_stepsize])
    full_converge = True
    
    for conv in conv_0[::i_stepsize]:
        if not conv[0]:
            full_converge = False
            break
        if not conv[1]:
            full_converge = False
            break
    
    if full_converge:
        print("full convergence")
        break

    for i in i_range_0:

        print("------- ", i, exc[i], inh[i])
        aln.params.mue_ext_mean = exc[i] * 5.
        aln.params.mui_ext_mean = inh[i] * 5.

        j = 1
        while cost_0[i][-j] == 0.:
            j += 1
            
        if j == cost_0[i].shape[0]-1:
            print("converged for ", i)
            if conv_0[i][0]:
                conv_0[i] = [True, True]
            else:
                conv_0[i] = [True, False]
            continue
    
        print("no convergence")

        weight_ = (factor_we * weights_0[i][1] * cost_uncontrolled[i] / cost_0[i][-j]
                       + factor_ws * weights_0[i][2] * cost_uncontrolled[i] / cost_0[i][-j]) - 1
        print("weight = ", weight_)
        cost.setParams(1.0, weight_ * factor_we, weight_ * factor_ws)

        setinit(initVars[i], aln)
        control0 = bestControl_0[i][:,:,n_pre-1:-n_post+1]

        # "HS", "FR", "PR", "HZ"
        cgv = None
        max_it = 500 * factor_iteration

        weights_0[i] = cost.getParams()

        bestControl_0[i], bestState_0[i], cost_0[i], runtime_0[i], grad_0[i], phi_0[i], costnode_0[i] = aln.A1(
            control0, target[i], c_scheme, u_mat, u_scheme, max_iteration_ = max_it, tolerance_ = tol,
            startStep_ = start_step, max_control_ = max_cntrl, min_control_ = min_cntrl, t_sim_ = dur,
            t_sim_pre_ = dur_pre, t_sim_post_ = dur_post, CGVar = cgv, control_variables_ = cntrl_vars_0,
            prec_variables_ = prec_vars, transition_time_ = trans_time)
        
    counter += 1
    

--------------- 0
[[True, True], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [True, True], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False]]
-------  11 0.4500000000000001 0.42500000000000016
no convergence
weight =  6009.80247808957
set cost params:  1.0 0.0 6009.80247808957
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  13308.079798010222
Gradient descend method:  None
RUN  1 , total integrated cost =  13308.074566569467
RUN  2 , total integrated cost =  13308.074054979244
RUN  3 , total integrated cost =  13308.073994771388
RUN  4 , total integrated cost =  13308.073980975336
RUN  5 , total integrated cost =  13308.073977889288
RUN  6 , total integrated cost =  13308.073977480586
RUN  7 , total integrated cost =  13308.073977427703
RUN  8 , total integrated cost =  13308.073977421562
RUN  9 , total integrated cost =  13308.073977420876
RUN  10 , total integrated

ERROR:root:Problem in initial value trasfer


Control only changes marginally.
RUN  13 , total integrated cost =  13308.07397742076
Improved over  13  iterations in  1.2473395708948374  seconds by  4.373726000039824e-05  percent.
Problem in initial value trasfer:  Vmean_exc -56.672120568747815 -56.672126209209274
-------  22 0.5000000000000002 0.4750000000000002
no convergence
weight =  12397.175178650074
set cost params:  1.0 0.0 12397.175178650074
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  21562.297103469824
Gradient descend method:  None
RUN  1 , total integrated cost =  21562.291610622564
RUN  2 , total integrated cost =  21562.29121354987
RUN  3 , total integrated cost =  21562.291165132498
RUN  4 , total integrated cost =  21562.291158709686
RUN  5 , total integrated cost =  21562.291157683805
RUN  6 , total integrated cost =  21562.291157520143
RUN  7 , total integrated cost =  21562.29115749082
RUN  8 , total integrated cost =  21562.291157486427
RUN  9 , total integrated cost =  21562.29115748

ERROR:root:Problem in initial value trasfer


RUN  14 , total integrated cost =  21562.29115748557
Control only changes marginally.
RUN  14 , total integrated cost =  21562.29115748557
Improved over  14  iterations in  1.9484632145613432  seconds by  2.757583862944557e-05  percent.
Problem in initial value trasfer:  Vmean_exc -56.69836282188823 -56.698363752206674
-------  33 0.5000000000000002 0.5250000000000002
no convergence
weight =  7105.449603695533
set cost params:  1.0 0.0 7105.449603695533
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  21065.870412819862
Gradient descend method:  None
RUN  1 , total integrated cost =  21065.869705305642
RUN  2 , total integrated cost =  21065.869684139718
RUN  3 , total integrated cost =  21065.869683309364
RUN  4 , total integrated cost =  21065.8696832863
RUN  5 , total integrated cost =  21065.869683285066
RUN  6 , total integrated cost =  21065.869683285007
RUN  7 , total integrated cost =  21065.869683285004
State only changes marginally.


...

RUN  2 , total integrated cost =  13309.221377292646
Control only changes marginally.
RUN  2 , total integrated cost =  13309.221377292646
Improved over  2  iterations in  0.37368584983050823  seconds by  4.263256414560601e-14  percent.
Problem in initial value trasfer:  Vmean_exc -56.672120546322276 -56.672126187315044
-------  22 0.5000000000000002 0.4750000000000002
converged for  22
-------  33 0.5000000000000002 0.5250000000000002
converged for  33
-------  44 0.47500000000000014 0.5750000000000003
converged for  44
-------  55 0.4250000000000001 0.6250000000000003
converged for  55
-------  66 0.5250000000000001 0.6500000000000004
converged for  66
-------  88 0.5500000000000003 0.7250000000000004
no convergence
weight =  9560.418367204478
set cost params:  1.0 0.0 9560.418367204478
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  29122.750456279187
Gradient descend method:  None


ERROR:root:Problem in initial value trasfer


RUN  1 , total integrated cost =  29122.750456279187
Control only changes marginally.
RUN  1 , total integrated cost =  29122.750456279187
Improved over  1  iterations in  0.20342271216213703  seconds by  0.0  percent.
Problem in initial value trasfer:  Vmean_exc -56.70419512594293 -56.70419497855702
-------  99 0.4250000000000001 0.7750000000000005
converged for  99
-------  110 0.5000000000000002 0.8000000000000005
converged for  110
-------  121 0.5750000000000002 0.8250000000000005
converged for  121
-------  132 0.4500000000000001 0.8750000000000006
converged for  132
-------  143 0.5250000000000001 0.9000000000000006
converged for  143
--------------- 4
[[True, True], [False, False], [True, False], [True, False], [True, True], [True, True], [True, True], [True, True], [False, False], [True, True], [True, True], [True, True], [True, True], [True, True]]
-------  11 0.4500000000000001 0.42500000000000016
no convergence
weight =  6010.323965841857
set cost params:  1.0 0.0 6010.3239

ERROR:root:Problem in initial value trasfer


RUN  1 , total integrated cost =  13309.221377590324
Control only changes marginally.
RUN  1 , total integrated cost =  13309.221377590324
Improved over  1  iterations in  0.1999903842806816  seconds by  0.0  percent.
Problem in initial value trasfer:  Vmean_exc -56.672120546322276 -56.672126187315044
-------  22 0.5000000000000002 0.4750000000000002
converged for  22
-------  33 0.5000000000000002 0.5250000000000002
converged for  33
-------  44 0.47500000000000014 0.5750000000000003
converged for  44
-------  55 0.4250000000000001 0.6250000000000003
converged for  55
-------  66 0.5250000000000001 0.6500000000000004
converged for  66
-------  88 0.5500000000000003 0.7250000000000004
converged for  88
-------  99 0.4250000000000001 0.7750000000000005
converged for  99
-------  110 0.5000000000000002 0.8000000000000005
converged for  110
-------  121 0.5750000000000002 0.8250000000000005
converged for  121
-------  132 0.4500000000000001 0.8750000000000006
converged for  132
-------  1

In [24]:
print(conv_0[::i_stepsize])

with open(final_file,'wb') as f:
    pickle.dump([bestControl_0, bestState_0, cost_0, runtime_0, grad_0, phi_0,
                 costnode_0, weights_0], f)

[[True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True]]


In [25]:
bestControl_1 = [None] * len(exc)
bestState_1 = [None] * len(exc)
cost_1 = [None] * len(exc)
runtime_1 = [None] * len(exc)
grad_1 = [None] * len(exc)
phi_1 = [None] * len(exc)
costnode_1 = [None] * len(exc)
weights_1 = [None] * len(exc)

conv_1 = [[False]*2] * len(exc)

In [26]:
if os.path.isfile(final_file_1) :
    print("file found")
    
    with open(final_file_1,'rb') as f:
        load_array = pickle.load(f)

    bestControl_1 = load_array[0]
    bestState_1 = load_array[1]
    cost_1 = load_array[2]
    runtime_1 = load_array[3]
    grad_1 = load_array[4]
    phi_1 = load_array[5]
    costnode_1 = load_array[6]
    weights_1 = load_array[7]

file found


In [27]:
i_range_1 = range(0, len(exc),i_stepsize)
i_range_ = []

for i in i_range_1:
    if type(bestControl_1[i]) == type(None):
        i_range_.append(i)

i_range_1 = np.array(i_range_)  

print(i_range_1)

[ 11  22  33  44  55  66  88  99 110 121 132 143]


In [28]:
factor_iteration = 20

for i in i_range_1:        

    print("------- ", i, exc[i], inh[i])
    aln.params.mue_ext_mean = exc[i] * 5.
    aln.params.mui_ext_mean = inh[i] * 5.

    cost.setParams(1.0, 1. * factor_we, 1. * factor_ws)

    setinit(initVars[i], aln)
    control0 = bestControl_0[i][:,:,n_pre-1:-n_post+1]

    # "HS", "FR", "PR", "HZ"
    cgv = None
    max_it = int( 500 * factor_iteration )

    weights_1[i] = cost.getParams()

    bestControl_1[i], bestState_1[i], cost_1[i], runtime_1[i], grad_1[i], phi_1[i], costnode_1[i] = aln.A1(
        control0, target[i], c_scheme, u_mat, u_scheme, max_iteration_ = max_it, tolerance_ = tol,
        startStep_ = start_step, max_control_ = max_cntrl, min_control_ = min_cntrl, t_sim_ = dur,
        t_sim_pre_ = dur_pre, t_sim_post_ = dur_post, CGVar = cgv, control_variables_ = cntrl_vars_init,
        prec_variables_ = prec_vars, transition_time_ = trans_time)

-------  11 0.4500000000000001 0.42500000000000016
set cost params:  1.0 0.0 1.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  87.25255537555931
Gradient descend method:  None
RUN  1 , total integrated cost =  2.6353690648343435
RUN  2 , total integrated cost =  2.578953855318383
RUN  3 , total integrated cost =  2.544485768934876
RUN  4 , total integrated cost =  2.5073750855991896
RUN  5 , total integrated cost =  2.485124984383301
RUN  6 , total integrated cost =  2.4593767673936595
RUN  7 , total integrated cost =  2.4408917201472162
RUN  8 , total integrated cost =  2.415936726766549
RUN  9 , total integrated cost =  2.39871649418523
RUN  10 , total integrated cost =  2.3649806680413255
RUN  11 , total integrated cost =  2.3432176783924454
RUN  12 , total integrated cost =  2.290803007090707
RUN  13 , total integrated cost =  2.281006078354687
RUN  14 , total integrated cost =  2.2707543326699047
RUN  15 , total integrated cost =  2.268514316664386
RUN  1

RUN  18 , total integrated cost =  4.8905125109479055
RUN  19 , total integrated cost =  4.886608059396807
RUN  20 , total integrated cost =  4.886524568500721
RUN  30 , total integrated cost =  4.884677930871484
RUN  40 , total integrated cost =  4.8831445967550815
RUN  50 , total integrated cost =  4.8822640806430355
RUN  60 , total integrated cost =  4.881067612639416
RUN  70 , total integrated cost =  4.879778468800289
RUN  80 , total integrated cost =  4.878687351847155
RUN  90 , total integrated cost =  4.877596078717953
RUN  100 , total integrated cost =  4.877179262038515
RUN  110 , total integrated cost =  4.8762858145170584
RUN  120 , total integrated cost =  4.875733068200476
RUN  130 , total integrated cost =  4.875301498577485
RUN  140 , total integrated cost =  4.875168393000711
RUN  150 , total integrated cost =  4.874933710004683
RUN  160 , total integrated cost =  4.874306477219101
RUN  170 , total integrated cost =  4.873814009962895
RUN  180 , total integrated cost =

RUN  90 , total integrated cost =  3.076149488790194
RUN  100 , total integrated cost =  3.075833412404978
RUN  110 , total integrated cost =  3.0753754576833914
RUN  120 , total integrated cost =  3.0738362990900483
RUN  130 , total integrated cost =  3.070867220726254
RUN  140 , total integrated cost =  3.070857936635904
RUN  150 , total integrated cost =  3.070852208889775
RUN  160 , total integrated cost =  3.0708467039115885
RUN  170 , total integrated cost =  3.070841502070923
RUN  180 , total integrated cost =  3.070821693844634
RUN  190 , total integrated cost =  3.0708186830540596
RUN  200 , total integrated cost =  3.070786915187558
RUN  300 , total integrated cost =  3.069266860073272
Control only changes marginally.
RUN  316 , total integrated cost =  3.069266860070449
Improved over  316  iterations in  6.452058009803295  seconds by  97.36679166360481  percent.
-------  99 0.4250000000000001 0.7750000000000005
set cost params:  1.0 0.0 1.0
interpolate adjoint :  True True T

RUN  110 , total integrated cost =  2.4390654278697386
RUN  120 , total integrated cost =  2.4390368652644816
RUN  130 , total integrated cost =  2.439011394379351
RUN  140 , total integrated cost =  2.4389920865699986
RUN  150 , total integrated cost =  2.438964924559876
RUN  160 , total integrated cost =  2.438949670120416
RUN  170 , total integrated cost =  2.4388667300140305
RUN  180 , total integrated cost =  2.438856253654704
RUN  190 , total integrated cost =  2.43883660197355
RUN  200 , total integrated cost =  2.437765825777017
RUN  300 , total integrated cost =  2.4371712288041976
Control only changes marginally.
RUN  305 , total integrated cost =  2.4371712288041936
Improved over  305  iterations in  6.151905527338386  seconds by  98.46861752389681  percent.
-------  132 0.4500000000000001 0.8750000000000006
set cost params:  1.0 0.0 1.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  10.113513489116153
Gradient descend method:  None
RUN  1 , total in

ERROR:root:Problem in initial value trasfer


Control only changes marginally.
RUN  651 , total integrated cost =  7.7341824159825014
Improved over  651  iterations in  13.362937798723578  seconds by  23.526255990998706  percent.
Problem in initial value trasfer:  Vmean_exc -56.652329685668086 -56.652329670061775
-------  143 0.5250000000000001 0.9000000000000006
set cost params:  1.0 0.0 1.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  24.865949843404486
Gradient descend method:  None
RUN  1 , total integrated cost =  5.061732621084053
RUN  2 , total integrated cost =  5.054177971501783
RUN  3 , total integrated cost =  5.009013934830496
RUN  4 , total integrated cost =  4.978236107044887
RUN  5 , total integrated cost =  4.9769326857304605
RUN  6 , total integrated cost =  4.975406493105645
RUN  7 , total integrated cost =  4.974554526930478
RUN  8 , total integrated cost =  4.973391949569811
RUN  9 , total integrated cost =  4.972818540377271
RUN  10 , total integrated cost =  4.971877258705231
RUN  1

In [29]:
factor_iteration = 20
full_converge = False

for i in range(len(conv_1)):
    if i not in i_range_1:
        conv_1[i] = [True, True]
        
counter = 0

while full_converge == False:
    
    print('---------------', counter)
    
    print(conv_1[::i_stepsize])
    full_converge = True
    
    for conv in conv_1[::i_stepsize]:
        if not conv[0]:
            full_converge = False
            break
        if not conv[1]:
            full_converge = False
            break
    
    if full_converge:
        print("full convergence")
        break

    for i in i_range_1:        

        print("------- ", i, exc[i], inh[i])
        aln.params.mue_ext_mean = exc[i] * 5.
        aln.params.mui_ext_mean = inh[i] * 5.
        
        j = 1
        while cost_1[i][-j] == 0.:
            j += 1
            
        if j == cost_1[i].shape[0]-1:
            print("converged for ", i)
            if conv_1[i][0]:
                conv_1[i] = [True, True]
            else:
                conv_1[i] = [True, False]
            continue
    
        print("no convergence")
        
        cost.setParams(1.0, 1. * factor_we, 1. * factor_ws)

        setinit(initVars[i], aln)
        control0 = bestControl_1[i][:,:,n_pre-1:-n_post+1]

        # "HS", "FR", "PR", "HZ"
        cgv = None
        max_it = int( 500 * factor_iteration )

        weights_1[i] = cost.getParams()

        bestControl_1[i], bestState_1[i], cost_1[i], runtime_1[i], grad_1[i], phi_1[i], costnode_1[i] = aln.A1(
            control0, target[i], c_scheme, u_mat, u_scheme, max_iteration_ = max_it, tolerance_ = tol,
            startStep_ = start_step, max_control_ = max_cntrl, min_control_ = min_cntrl, t_sim_ = dur,
            t_sim_pre_ = dur_pre, t_sim_post_ = dur_post, CGVar = cgv, control_variables_ = cntrl_vars_init,
            prec_variables_ = prec_vars, transition_time_ = trans_time)
        
        counter += 1

--------------- 0
[[True, True], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [True, True], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False]]
-------  11 0.4500000000000001 0.42500000000000016
no convergence
set cost params:  1.0 0.0 1.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  2.2427744362267283
Gradient descend method:  None
RUN  1 , total integrated cost =  2.2427744362267275
RUN  2 , total integrated cost =  2.2427744362267275
Control only changes marginally.
RUN  2 , total integrated cost =  2.2427744362267275
Improved over  2  iterations in  0.09941302239894867  seconds by  4.263256414560601e-14  percent.
-------  22 0.5000000000000002 0.4750000000000002
no convergence
set cost params:  1.0 0.0 1.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  1.761212498929993
Gradient descend method:  None
RUN  1 , total integrated cost =  1.7

ERROR:root:Problem in initial value trasfer


RUN  1 , total integrated cost =  2.4371712288041936
Control only changes marginally.
RUN  1 , total integrated cost =  2.4371712288041936
Improved over  1  iterations in  0.06394718959927559  seconds by  0.0  percent.
-------  132 0.4500000000000001 0.8750000000000006
no convergence
set cost params:  1.0 0.0 1.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  7.7341824159825014
Gradient descend method:  None
RUN  1 , total integrated cost =  7.7341824159825014
Control only changes marginally.
RUN  1 , total integrated cost =  7.7341824159825014
Improved over  1  iterations in  0.06249438598752022  seconds by  0.0  percent.
Problem in initial value trasfer:  Vmean_exc -56.652329685668086 -56.652329670061775
-------  143 0.5250000000000001 0.9000000000000006
no convergence
set cost params:  1.0 0.0 1.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  4.755998486853164
Gradient descend method:  None
RUN  1 , total integrated cost =  4.755998

In [30]:
print(conv_1[::i_stepsize])

with open(final_file_1,'wb') as f:
    pickle.dump([bestControl_1, bestState_1, cost_1, runtime_1, grad_1, phi_1,
                 costnode_1, weights_1], f)

[[True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True]]
